In [82]:
import sys, os
import pandas as pd
import numpy as np
from pathlib import Path

# Przejdź do katalogu głównego projektu tylko raz — bezpieczne przy wielokrotnym uruchomieniu
if os.path.basename(os.getcwd()) != "analiza_sprzedazy_w_sklepie":
    os.chdir("../..")

sys.path.insert(0, os.getcwd())
import utils.tools as tools

In [83]:
# Suma kontrolna (hash) danych w pliku Parquet, z uwzględnieniem sortowania po kolumnach TowId i Data.

nazwa_pliku = "fact_inka_soft.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")


Mój hash (posortowane):   fact_inka_soft.parquet   87ff0e55ac94368b71e9e36a33c43e321c2bfaeeaffae189daa9d0d8996cff10


In [84]:
df_kalendarz = pd.read_parquet(f"dane/interim/{nazwa_pliku}")


In [85]:
df_kalendarz['WplywNaStan'].info()

<class 'pandas.Series'>
RangeIndex: 4057332 entries, 0 to 4057331
Series name: WplywNaStan
Non-Null Count    Dtype
--------------    -----
4057332 non-null  bool 
dtypes: bool(1)
memory usage: 3.9 MB


In [86]:


df_sprzedaz = df_kalendarz[df_kalendarz['TypDok'] == 21].copy()

towid_do_kalendarza = df_sprzedaz['TowId'].unique()
print(f"TowId z jakąkolwiek sprzedażą: {len(towid_do_kalendarza):,}")

data_min = df_kalendarz['Data'].min()
data_max = df_kalendarz['Data'].max()
kalendarz_dni = pd.date_range(start=data_min, end=data_max, freq='D')
print(f"Dni w zakresie: {len(kalendarz_dni):,}")

siatka = pd.MultiIndex.from_product(
    [towid_do_kalendarza, kalendarz_dni], names=['TowId', 'Data']
).to_frame(index=False)
print(f"Rozmiar siatki: {len(siatka):,}")

pelny_kalendarz = siatka.merge(df_sprzedaz, on=['TowId', 'Data'], how='left')
print(f"Rozmiar po scaleniu: {len(pelny_kalendarz):,}")


TowId z jakąkolwiek sprzedażą: 12,481
Dni w zakresie: 1,127
Rozmiar siatki: 14,066,087
Rozmiar po scaleniu: 15,810,161


In [87]:

kolumny_towid = ['NazwaTow', 'EAN', 'AsId', 'Producent', 'NazwaAsort']

atrybuty_towid = df_kalendarz[['TowId'] + kolumny_towid].drop_duplicates(subset='TowId')

for kol in kolumny_towid:
    mapa = atrybuty_towid.set_index('TowId')[kol]
    pelny_kalendarz[kol] = pelny_kalendarz[kol].fillna(pelny_kalendarz['TowId'].map(mapa))

pelny_kalendarz['DokId'] = pelny_kalendarz['DokId'].fillna(-1).astype(int)


# Zakres życia per TowId (z bufor 14 dni,)
daty_sku = df_sprzedaz.groupby('TowId')['Data'].agg(DataStart='min', DataKoniec='max').reset_index()
daty_sku['DataKoniec'] = (daty_sku['DataKoniec'] + pd.Timedelta(days=14)).clip(upper=data_max)

# Dołączamy do pelny_kalendarz i filtrujemy
pelny_kalendarz = pelny_kalendarz.merge(daty_sku, on='TowId', how='left')

przed = len(pelny_kalendarz)
pelny_kalendarz = pelny_kalendarz[
    (pelny_kalendarz['Data'] >= pelny_kalendarz['DataStart']) &
    (pelny_kalendarz['Data'] <= pelny_kalendarz['DataKoniec'])
].copy()

print(f"Przed przycięciem: {przed:,} wierszy")
print(f"Po przycięciu:     {len(pelny_kalendarz):,} wierszy")

# Opcjonalnie: usuń pomocnicze kolumny DataStart/DataKoniec, jeśli nie chcesz ich w finalnym pliku
pelny_kalendarz = pelny_kalendarz.drop(columns=['DataStart', 'DataKoniec'])




print(f"\nPuste rekordy (DokId=-1): {(pelny_kalendarz['DokId']==-1).sum():,}")
print(f"Realne rekordy sprzedaży: {(pelny_kalendarz['DokId']!=-1).sum():,}")


Przed przycięciem: 15,810,161 wierszy
Po przycięciu:     8,802,515 wierszy

Puste rekordy (DokId=-1): 5,428,540
Realne rekordy sprzedaży: 3,373,975


In [88]:
# Uzupełnianie brakujące wartości w kolumnach
maska_puste = pelny_kalendarz['DokId'] == -1

pelny_kalendarz.loc[maska_puste, 'Kolejnosc'] = pelny_kalendarz.loc[maska_puste, 'Kolejnosc'].fillna(0)
pelny_kalendarz.loc[maska_puste, 'NrPozycji'] = pelny_kalendarz.loc[maska_puste, 'NrPozycji'].fillna(0)
pelny_kalendarz.loc[maska_puste, 'TypPoz'] = pelny_kalendarz.loc[maska_puste, 'TypPoz'].fillna(4)
pelny_kalendarz.loc[maska_puste, 'IloscPlus'] = pelny_kalendarz.loc[maska_puste, 'IloscPlus'].fillna(0)
pelny_kalendarz.loc[maska_puste, 'IloscMinus'] = pelny_kalendarz.loc[maska_puste, 'IloscMinus'].fillna(0)
pelny_kalendarz['CenaPoRab'] = pelny_kalendarz.groupby('TowId')['CenaPoRab'].ffill()
pelny_kalendarz.loc[maska_puste, 'Wartosc'] = pelny_kalendarz.loc[maska_puste, 'Wartosc'].fillna(0)
pelny_kalendarz.loc[maska_puste, 'KolejnyWDniu'] = pelny_kalendarz.loc[maska_puste, 'KolejnyWDniu'].fillna(-1)
pelny_kalendarz.loc[maska_puste, 'NrDok'] = pelny_kalendarz.loc[maska_puste, 'NrDok'].fillna('brak')
pelny_kalendarz.loc[maska_puste, 'TypDok'] = pelny_kalendarz.loc[maska_puste, 'TypDok'].fillna(21)
pelny_kalendarz.loc[maska_puste, 'AktywnyDok'] = pelny_kalendarz.loc[maska_puste, 'AktywnyDok'].fillna(1)
pelny_kalendarz.loc[maska_puste, 'Razem'] = pelny_kalendarz.loc[maska_puste, 'Razem'].fillna(0)
pelny_kalendarz.loc[maska_puste, 'DoZaplaty'] = pelny_kalendarz.loc[maska_puste, 'DoZaplaty'].fillna(0)
pelny_kalendarz.loc[maska_puste, 'Zaplacono'] = pelny_kalendarz.loc[maska_puste, 'Zaplacono'].fillna(0)
pelny_kalendarz.loc[maska_puste, 'Opis1'] = pelny_kalendarz.loc[maska_puste, 'Opis1'].fillna("brak")

pelny_kalendarz.loc[maska_puste, 'Producent'] = pelny_kalendarz.loc[maska_puste, 'Producent'].fillna(-1)
pelny_kalendarz.loc[maska_puste, 'AktywnyTow'] = pelny_kalendarz.loc[maska_puste, 'AktywnyTow'].fillna(1)
pelny_kalendarz.loc[maska_puste, 'Dokument'] = pelny_kalendarz.loc[maska_puste, 'Dokument'].fillna("DF")

pelny_kalendarz.loc[maska_puste, 'MetodaLiczenia'] = pelny_kalendarz.loc[maska_puste, 'MetodaLiczenia'].fillna('IP')
pelny_kalendarz.loc[maska_puste, 'Mnoznik'] = pelny_kalendarz.loc[maska_puste, 'Mnoznik'].fillna(0)
pelny_kalendarz.loc[maska_puste, 'TypRuchu'] = pelny_kalendarz.loc[maska_puste, 'TypRuchu'].fillna('sprzedaz')
pelny_kalendarz['CzyNiechciane'] = pelny_kalendarz['CzyNiechciane'].fillna(0).astype(int)
pelny_kalendarz['WplywNaStan']   = pelny_kalendarz['WplywNaStan'].fillna(0).astype(int)


In [89]:
# Pozostałe ruchy magazynowe (PZ, REM, BO, ST, ROZB, PRZES...) — bez przycinania dat
ruchy_pozostale = df_kalendarz[
    (df_kalendarz['TypDok'] != 21) &
    (df_kalendarz['TowId'].isin(towid_do_kalendarza))
].copy()

# ten sam typ co w pelny_kalendarz (inaczej mieszane typy przy zapisie do parquet)
ruchy_pozostale['CzyNiechciane'] = ruchy_pozostale['CzyNiechciane'].astype(int)
ruchy_pozostale['WplywNaStan']   = ruchy_pozostale['WplywNaStan'].astype(int)

pelny_kalendarz = pd.concat([pelny_kalendarz, ruchy_pozostale], ignore_index=True)
pelny_kalendarz = pelny_kalendarz.sort_values(['TowId', 'Data', 'DokId', 'Kolejnosc']).reset_index(drop=True)

puste = (pelny_kalendarz['DokId'] == -1).sum()
print(f"Dołączone ruchy: {len(ruchy_pozostale):,}  (oczekiwane 683 357)")
print(f"Razem: {len(pelny_kalendarz):,}  (oczekiwane {len(df_kalendarz) + puste:,})")
assert len(pelny_kalendarz) == len(df_kalendarz) + puste

Dołączone ruchy: 683,357  (oczekiwane 683 357)
Razem: 9,485,872  (oczekiwane 9,485,872)


In [90]:
# Kontrola: żadna transakcja nie mogła zginąć ani się zduplikować przy scalaniu
print(f"Wiersze w df_kalendarz (surowe): {len(df_kalendarz):,}")
print(f"Realne rekordy w pelny_kalendarz: {(pelny_kalendarz['DokId']!=-1).sum():,}")
print(f"Zgodność: {(pelny_kalendarz['DokId']!=-1).sum() == len(df_kalendarz)}")


Wiersze w df_kalendarz (surowe): 4,057,332
Realne rekordy w pelny_kalendarz: 4,057,332
Zgodność: True


In [91]:
pelny_kalendarz.to_parquet(
    "dane/interim/fact_inka_calendar.parquet",
    compression='zstd',
    index=False
)


In [92]:
# Suma kontrolna
nazwa_pliku = "fact_inka_calendar.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")


Mój hash (posortowane):   fact_inka_calendar.parquet   dd831a813d662099be81de7050891487f7e95808cf0c2aad19404e7a4484ef6e


In [93]:
# hash pliku kalendarz_pelny_towid.parquet:  dd831a813d662099be81de7050891487f7e95808cf0c2aad19404e7a4484ef6e